In [1]:
import numpy as np

from typing import Union, List
from pymatching import Matching
import operators as o
import decoding.nine_bit_rep_decoding as nr
import decoding.three_bit_rep_decoding as tr
import decoding.utils as d

In [2]:
rels = {
    ('X', 'Z'): 'Y',
    ('Z', 'X'): 'Y',
    ('X', 'Y'): 'Z',
    ('Y', 'X'): 'Z',
    ('Y', 'Z'): 'X',
    ('Z', 'Y'): 'X',
}

# TODO: Check assumptions
def get_syndrome_bit(state, stabilizer):
    bit = 0
    for comp in zip(state, stabilizer):
        if 'I' not in comp:
            if comp[0] != comp[1]:
                bit += 1

    return bit % 2

In [3]:
rounds = 20
shots = 1000
px = 0.04
pm = 0.00

intial_state = o.Operator(3)

S1 = o.Z(3, [0, 1])
S2 = o.Z(3, [1, 2])

matching_graph = tr.build_3_qubit_matching_graph(rounds, px, pm)

result = 0
for shot in range(shots):
    syndrome = []
    true_phys_error = np.zeros(3)
    state = intial_state.copy()
    for rnd in range(rounds):

        
        errors = np.random.random(3) < px
        x_indices = [idx for idx, i in enumerate(errors) if i]
        
        state = state * o.X(3, x_indices)
        # print(state)

        true_phys_error[x_indices] += 1

        sb1 = get_syndrome_bit(state, S1)
        sb2 = get_syndrome_bit(state, S2)

        syndrome.append([sb1, sb2])

    
    detections = d.detect_changes(np.array(syndrome))
    # print(detections)
    matches = d.match_errors(matching_graph, detections)
    predicted_errors = tr.extract_3_qubit_predicted_errors(matches)
    # print('True', true_phys_error)
    # print('Pred', predicted_errors)

    true_total_error = [i % 2 for i in true_phys_error]
    
    diff = sum(np.abs(true_total_error - predicted_errors))
    
    if diff != 0:
        result += 1

print(result/shots)
        

0.079


In [10]:
rounds = 20
shots = 2000
px = 0.027
pz = 0.027
pm = 0.00

intial_state = o.Operator(9)

stabs = {
    'S1': o.Z(9, [0, 1]),
    'S2': o.Z(9, [1, 2]),
    'S3': o.Z(9, [3, 4]),
    'S4': o.Z(9, [4, 5]),
    'S5': o.Z(9, [6, 7]),
    'S6': o.Z(9, [7, 8]),
    'S7': o.X(9, [0, 1, 2, 3, 4, 5]),
    'S8': o.X(9, [3, 4, 5, 6, 7, 8]),
    }

num_stabilizers = 8

matching_graph = nr.build_9_qubit_matching_graph(rounds, px, pz, pm)

result = 0
for shot in range(shots):
    syndrome = []
    true_phys_error = {
        'X': np.zeros(9),
        'Z': np.zeros(9)
    }
    
    state = intial_state.copy()
    for rnd in range(rounds):
        
        # ERRORS
        # TODO: can do this more efficiently   
        x_errors = np.random.random(9) < px
        z_errors = np.random.random(9) < pz
        x_indices, z_indices = [idx for idx, i in enumerate(x_errors) if i], [idx for idx, i in enumerate(z_errors) if i]
        
        state = state * o.X(9, x_indices)
        state = state * o.Z(9, z_indices)
        # print(state)
        
        true_phys_error['X'][x_indices] += 1
        true_phys_error['Z'][z_indices] += 1
        # print(z_indices)

        t_measurements = []
        for stab in stabs.keys():
            t_measurements.append(get_syndrome_bit(state, stabs[stab]))
        
        syndrome.append(t_measurements)

    # print(syndrome)
    detections = d.detect_changes(np.array(syndrome))
    # print(detections)
    matches = d.match_errors(matching_graph, detections)
    # print(matches)
    predicted_x_errors, predicted_z_errors = nr.extract_9_qubit_predicted_errors(matches)
    # print(predicted_x_errors, predicted_z_errors)

    true_total_error_x = [i % 2 for i in true_phys_error['X']]

    # print('True', true_phys_error['Z'])
    true_total_error_z = [sum(true_phys_error['Z'][i:i+3]) for i in range(0, len( true_phys_error['Z']), 3)]
    true_total_error_z = [i % 2 for i in true_total_error_z]

    # print(true_total_error_x, true_total_error_z)
    
    diff_x = [int(a) ^ int(b) for a, b in zip(true_total_error_x, predicted_x_errors)]
    diff_z = [int(a) ^ int(b) for a, b in zip(true_total_error_z, predicted_z_errors)]
    # print(sum(diff_x), sum(diff_z))

    # TODO: not differentiating between logical errors and non-code states. Individual block can be decoded with 'logical error' (IIIXXXIII)
    if (sum(diff_x) != 0) | (sum(diff_z) != 0):
        result += 1

print(result/shots)

0.342


In [ ]:
#Surface code
3*3

data_qubits = o.Operator(2)

# Maybe dont even need an initial state? Maybe don't need to measure stabilizers
# Can just directly add noise and generate what that would look like in the detectors/syndrome
# Generate blank syndrome
# [[0, 0, 0,]] 
# [[0, 0, 0,]]
# [[0, 0, 0,]]

# [[1, 1, 0,]] z11
# [[0, 0, 1,]] z10
# [[0, 0, 1,]]

In [9]:
0.92**4

0.7163929600000001

In [14]:
0.92**3 * 0.6245

0.4862906560000001